# 10 · Production model for the 3D map: the frozen model refitted on all 120 Downtown boreholes

**Decision (2026-10-27, before the new external boreholes were evaluated):** the 3D map uses the final model refitted on **all 120 Downtown boreholes**: the 109 development boreholes and the 11 boreholes of the first external holdout, which has served its purpose. Nothing else changes:

* same model family and the **same hyperparameters, filled-cell weight and threshold** as the frozen model (no new tuning);
* same cells (0.5 m, 0–15 m, gap-filled) and inputs.

This refit is **not evaluated**: its expected accuracy is the one estimated for the frozen model (spatial CV, notebook 04; external tests, notebooks 05 and 09). More data near the 11 added boreholes can only help the map there. The reliability zones are recomputed against all 120 boreholes.

In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

import joblib
from siemreap import nested_cv as NCV, grid3d, geo
AGG = config.stage_dir(cfg, "10_map_model", "aggregate")
RES = config.stage_dir(cfg, "10_map_model", "restricted")
mt = pd.read_csv(config.processed_dir(cfg) / "cells_all.csv")
frozen = json.loads((config.stage_dir(cfg, "05_final", "aggregate") / "final_model.json").read_text())
print(json.dumps({k: frozen[k] for k in ("model", "block_m", "params", "threshold", "model_sha256")}, indent=1, default=str))

## 1. Refit on all 120 Downtown boreholes

In [ ]:
train = mt[mt.is_downtown & mt.set.isin(["development", "external"])]
print(f"{train.BH_Code.nunique()} boreholes, {len(train)} cells ({int((train.measured == 0).sum())} filled)")
model = NCV.refit(train, cfg, frozen["model"], frozen["params"], seed_tag=f"map120_{frozen['model']}", mode=MODE)
path = RES / f"map_model_{frozen['model']}_120bh.joblib"
joblib.dump(model, path, compress=3)
rec = {"model": frozen["model"], "params": frozen["params"], "threshold": frozen["threshold"],
       "trained_on_boreholes": int(train.BH_Code.nunique()), "trained_on_cells": int(len(train)),
       "source_frozen_model_sha256": frozen["model_sha256"], "model_file": path.name,
       "model_sha256": config.sha256_file(path), "evaluated": False, "mode": MODE}
(AGG / "map_model.json").write_text(json.dumps(rec, indent=1, default=str))
print("saved", path.name, rec["model_sha256"][:16])

## 2. 3D prediction

Columns every `SPACING` metres inside the Downtown polygon (50 m: about half the 106 m autocorrelation range; see the design report), layers of 0.5 m from 0 to 15 m. Reliability zones by distance to the nearest of the 120 boreholes: 1 ≤ 100 m, 2 ≤ 300 m, 3 > 300 m.

In [ ]:
SPACING = 50.0     # m; 25.0 gives the finer grid of notebook 05
gcfg = {**cfg, "grid3d": {**cfg["grid3d"], "spacing_m": cfg["smoke"]["grid_spacing_m"] if MODE == "smoke" else SPACING}}
cols = grid3d.columns(gcfg)
grid = grid3d.cells(cols, gcfg)
grid = grid3d.predict(model, frozen["threshold"], grid, cfg["grid3d"]["chunk_rows"])
xy120 = train.drop_duplicates("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]].values
elev = (float(train.Top_Elevation_m.min()), float(train.Top_Elevation_m.max()))
grid = grid3d.add_flags(grid, cols, xy120, cfg["grid3d"]["zone_edges_m"][-1], elev, cfg["grid3d"]["zone_edges_m"])
share = {"columns": int(len(cols)), "cells": int(len(grid)), "spacing_m": gcfg["grid3d"]["spacing_m"],
         "reliability_zone_share": grid.reliability_zone.value_counts(normalize=True).sort_index().round(4).to_dict(),
         "vertical_extrapolation": float(grid.extrap_vertical.mean()), "predicted_clay_share": float(grid.clay.mean())}
print(json.dumps(share, indent=1))
tag = f"{int(gcfg['grid3d']['spacing_m'])}m"
grid3d.to_points_csv(grid, RES / f"siemreap_3d_points_120bh_{tag}.csv")
try:
    vox = grid3d.to_voxel_netcdf(grid, cols, gcfg, RES / f"siemreap_3d_voxels_120bh_{tag}.nc")
except ImportError as e:
    vox = {"skipped": str(e)}
(AGG / f"prediction_3d_summary_120bh_{tag}.json").write_text(json.dumps({**share, "voxel": vox, **rec}, indent=1, default=str))

In [ ]:
# Quick look (restricted figure: Downtown extent)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), dpi=150, sharey=True)
for ax, d in zip(axes, [1.0, 7.0, 13.0]):
    g = grid[np.isclose(grid.depth_top_m, d)]
    sc_ = ax.scatter(g.UTM_Easting_m, g.UTM_Northing_m, c=g.p_clay, cmap="RdBu", vmin=0, vmax=1, s=2 if MODE == "full" else 6, marker="s")
    z3 = g[g.reliability_zone == 3]
    ax.scatter(z3.UTM_Easting_m, z3.UTM_Northing_m, s=0.3, color="k", alpha=0.12)
    ax.set_title(f"P(Clay), {d:g}-{d + 0.5:g} m (dots: > 300 m from a borehole)", fontsize=9, loc="left")
    ax.set_aspect("equal"); ax.ticklabel_format(useOffset=False, style="plain")
fig.colorbar(sc_, ax=axes, shrink=0.8, label="P(Clay): blue = Clay, red = Sand")
fig.savefig(RES / "fig_3d_depth_slices_120bh.png", bbox_inches="tight")

### ArcGIS Pro / QGIS

Same as notebook 05, with the files `siemreap_3d_points_120bh_50m.csv` and `siemreap_3d_voxels_120bh_50m.nc` in `outputs/10_map_model/restricted/`. Show `reliability_zone` on every map.